# exp016 — Entrada 2.5D (cortes vecinos como RGB)

SPEC: `SPEC.md` en esta carpeta. Receta base del equipo: DINOv2-small por corte, atención sobre cortes y series, backbone lr 1e-5, 8 épocas.
Split canónico, semillas fijas y el entrenamiento completo en este notebook.

Celdas fijas (no cambies el orden): **0** parámetros · **1** setup · **2** datos y split · **3** modelo ·
**4** entrenamiento · **5** evaluación · **6** guardar · **7** verdict.
Para un experimento nuevo solo se cambian las celdas que dice el PLAN de su SPEC.

## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp016_entrada_25d"
PADRE = "exp002"
FOLDS = [0]
SEED = 42
SPLIT = "split_v1_seed42"
SELECT_ON = "pseudo"
SMOKE = True

LABELS_CSV = "/workspace/data/labels/report_labels_v1.csv"
CACHE_INDEX = "/workspace/data/cache/v1_256/index.csv"
BACKBONE_PATH = "/workspace/models/dinov2-small"

IMAGE_SIZE = 224
MAX_SERIES = 5
N_SLICES = 16
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 8
FREEZE_EPOCHS = 1
LR = 1e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
N_WORKERS = 8
NOTES = "referencia del equipo sobre split_v1"

## 1. Setup

In [ ]:
import json
import math
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.data import KneeDataset
from src.knee.experiments import decision, git_info, guardar_metricas, leer_metricas, seed_everything
from src.knee.metrics import mean_auc
from src.knee.model import KneeModel
from src.knee.splits import load_split

assert SELECT_ON == "pseudo", "Gold nunca selecciona checkpoints (docs/06_SDD.md)"
assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe la SPEC del experimento"
AUTHOR = current_user()
RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if SMOKE:
    EPOCHS, FREEZE_EPOCHS = 1, 0
seed_everything(SEED)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · {torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu'}")
if GIT["src_dirty"]:
    print("OJO: src/ o SPLITS/ tienen cambios sin commit; el resultado no será reproducible.")

## 2. Datos y split (las revisiones detienen el notebook si algo está mal)

In [ ]:
split = load_split(SPLIT)
pseudo = pd.read_csv(LABELS_CSV)
pseudo["StudyInstanceUID"] = pseudo.StudyInstanceUID.astype(str)
pseudo = pseudo[["StudyInstanceUID", *LABELS]]
train_csv = pd.read_csv("/workspace/data/raw/train.csv", encoding="utf-8")
gold = train_csv.dropna(subset=LABELS, how="all")[["StudyInstanceUID", *LABELS]]
gold["StudyInstanceUID"] = gold.StudyInstanceUID.astype(str)

pool = split[split.fold >= 0].merge(pseudo, on="StudyInstanceUID", how="inner")
assert len(gold) == 58 and set(gold.StudyInstanceUID) == set(split.loc[split.is_gold == 1, "StudyInstanceUID"])
assert not set(gold.StudyInstanceUID) & set(pool.StudyInstanceUID), "gold dentro del entrenamiento"
assert set(FOLDS) <= set(range(split.attrs["n_folds"]))
print(f"pool {len(pool)} estudios · gold {len(gold)} · folds {pool.fold.value_counts().sort_index().to_dict()}")

import cv2
import src.knee.data as knee_data

_prepare_slices_parent = knee_data.prepare_slices

def prepare_slices_25d(volume, n_slices, image_size, train=False):
    """exp016: igual que prepare_slices del padre, pero los 3 canales son los cortes (i-1, i, i+1) en lugar de i x3.
    En los bordes se repite el corte del borde. Mismos sorteos aleatorios y en el mismo orden que el padre;
    la misma matriz afín, gain y offset para los 3 canales."""
    indices = knee_data.sample_slice_indices(len(volume), n_slices, train)
    last = len(volume) - 1
    resized = {}

    def resize(i):
        if i not in resized:
            resized[i] = cv2.resize(volume[i], (image_size, image_size), interpolation=cv2.INTER_AREA)
        return resized[i]

    images = np.stack([
        np.stack([resize(int(min(max(index + d, 0), last))) for d in (-1, 0, 1)])
        for index in indices
    ]).astype(np.float32)  # [S, 3, H, W]
    if train:
        gain = np.random.uniform(0.9, 1.1)
        offset = np.random.uniform(-8, 8)
        angle = np.random.uniform(-5, 5)
        scale = np.random.uniform(0.97, 1.03)
        matrix = cv2.getRotationMatrix2D((image_size / 2, image_size / 2), angle, scale)
        images = np.stack([
            np.stack([cv2.warpAffine(channel, matrix, (image_size, image_size), borderMode=cv2.BORDER_REFLECT_101)
                      for channel in image])
            for image in images
        ])
        images = np.clip(images * gain + offset, 0, 255)
    images = images / 255.0
    mean = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)[None, :, None, None]
    std = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)[None, :, None, None]
    return torch.from_numpy((images - mean) / std).float()

class KneeDataset25D(KneeDataset):
    """KneeDataset del padre sin cambios (series, cortes, etiquetas, metadatos); solo cambia el armado de canales."""
    def __getitem__(self, index):
        knee_data.prepare_slices = prepare_slices_25d
        try:
            return super().__getitem__(index)
        finally:
            knee_data.prepare_slices = _prepare_slices_parent

_cache = pd.read_csv(CACHE_INDEX)
_spacing_cols = [c for c in _cache.columns if "spacing" in c.lower() or "thickness" in c.lower()]
print("espaciado entre cortes del cache:", {c: round(float(pd.to_numeric(_cache[c], errors="coerce").mean()), 3) for c in _spacing_cols} or "sin columna en el índice")

dataset_args = dict(cache_index=CACHE_INDEX, max_series=MAX_SERIES, n_slices=N_SLICES, image_size=IMAGE_SIZE)
loader_args = dict(batch_size=BATCH_SIZE, num_workers=N_WORKERS, pin_memory=True, persistent_workers=N_WORKERS > 0)

def make_loaders(fold):
    train_ids = pool.loc[pool.fold != fold, "StudyInstanceUID"]
    val_ids = pool.loc[pool.fold == fold, "StudyInstanceUID"]
    gold_ids = gold.StudyInstanceUID
    if SMOKE:
        train_ids, val_ids, gold_ids = train_ids[:80], val_ids[:40], gold_ids
    generator = torch.Generator().manual_seed(SEED + fold)
    return (
        DataLoader(KneeDataset25D(train_ids, labels=pool, train=True, **dataset_args), shuffle=True, generator=generator, **loader_args),
        DataLoader(KneeDataset25D(val_ids, labels=pool, train=False, **dataset_args), shuffle=False, **loader_args),
        DataLoader(KneeDataset25D(gold_ids, labels=gold, train=False, **dataset_args), shuffle=False, **loader_args),
    )

## 3. Modelo

In [ ]:
def build_model():
    model = KneeModel(backbone_path=BACKBONE_PATH, gradient_checkpointing=True)
    return model.to(DEVICE)

def build_optimizer(model, steps_per_epoch):
    backbone_ids = {id(p) for p in model.backbone.parameters()}
    optimizer = AdamW([
        {"params": list(model.backbone.parameters()), "lr": BACKBONE_LR},
        {"params": [p for p in model.parameters() if id(p) not in backbone_ids], "lr": LR},
    ], weight_decay=WEIGHT_DECAY)
    total = max(1, steps_per_epoch * EPOCHS)
    warmup = int(total * WARMUP_RATIO)

    def schedule(step):
        if step < warmup:
            return step / max(1, warmup)
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))

    return optimizer, torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)

def masked_bce(logits, targets):
    valid = torch.isfinite(targets)
    if not valid.any():
        return logits.sum() * 0.0
    losses = torch.nn.functional.binary_cross_entropy_with_logits(logits, torch.nan_to_num(targets), reduction="none")
    return losses[valid].mean()

## 4. Entrenamiento

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    predictions, targets, studies = [], [], []
    for batch in loader:
        with model.autocast_context(DEVICE):
            logits = model(batch["images"].to(DEVICE), batch["series_meta"].to(DEVICE), batch["series_mask"].to(DEVICE))
        predictions.append(logits.sigmoid().float().cpu().numpy())
        targets.append(batch["targets"].numpy())
        studies.extend(batch["study"])
    predictions, targets = np.concatenate(predictions), np.concatenate(targets)
    frame = pd.DataFrame(predictions, columns=[f"pred_{label}" for label in LABELS])
    frame.insert(0, "StudyInstanceUID", studies)
    return frame, mean_auc(targets, predictions)

def train_fold(fold):
    seed_everything(SEED + fold)
    train_loader, val_loader, gold_loader = make_loaders(fold)
    model = build_model()
    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    optimizer, scheduler = build_optimizer(model, steps_per_epoch)
    history, best = [], None
    start = time.time()
    for epoch in range(EPOCHS):
        model.freeze_backbone(epoch < FREEZE_EPOCHS)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        for step, batch in enumerate(train_loader):
            with model.autocast_context(DEVICE):
                logits = model(batch["images"].to(DEVICE, non_blocking=True),
                               batch["series_meta"].to(DEVICE, non_blocking=True),
                               batch["series_mask"].to(DEVICE, non_blocking=True))
                loss = masked_bce(logits, batch["targets"].to(DEVICE, non_blocking=True)) / GRAD_ACCUM
            loss.backward()
            running += loss.item() * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        oof, pseudo_metrics = predict(model, val_loader)
        gold_preds, gold_metrics = predict(model, gold_loader)
        history.append({"epoch": epoch + 1, "train_loss": running / len(train_loader),
                        "pseudo_auc": pseudo_metrics["mean"], "gold_auc": gold_metrics["mean"],
                        "minutes": (time.time() - start) / 60})
        pd.DataFrame(history).to_csv(RESULTS / f"history_f{fold}.csv", index=False)
        print(f"fold {fold} epoch {epoch + 1}: loss {history[-1]['train_loss']:.4f} "
              f"pseudo {pseudo_metrics['mean']:.4f} gold {gold_metrics['mean']:.4f} ({history[-1]['minutes']:.1f} min)")
        if best is None or pseudo_metrics["mean"] > best["pseudo"]["mean"]:
            best = {"epoch": epoch + 1, "pseudo": pseudo_metrics, "gold": gold_metrics}
            torch.save({"model": model.state_dict(), "exp": EXP, "fold": fold, "epoch": epoch + 1,
                        "params": {k: v for k, v in PARAMS.items()}}, RESULTS / f"best_f{fold}.pt")
            oof.to_csv(RESULTS / f"oof_f{fold}.csv", index=False)
            gold_preds.to_csv(RESULTS / f"gold_f{fold}.csv", index=False)
    return {"pseudo_auc": best["pseudo"]["mean"], "gold_auc": best["gold"]["mean"], "best_epoch": best["epoch"],
            "pseudo_per_label": best["pseudo"]["per_label"], "gold_per_label": best["gold"]["per_label"],
            "gpu_hours": (time.time() - start) / 3600, "ckpt_path": str(RESULTS / f"best_f{fold}.pt")}

PARAMS = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, bool))}
per_fold = {}
for fold in FOLDS:
    per_fold[str(fold)] = train_fold(fold)

## 5. Evaluación contra el padre

In [ ]:
gold_auc = mean_auc(
    gold.set_index("StudyInstanceUID").loc[:, LABELS].to_numpy(),
    np.mean([pd.read_csv(RESULTS / f"gold_f{f}.csv").set_index("StudyInstanceUID").loc[gold.StudyInstanceUID].to_numpy()
             for f in FOLDS], axis=0),
)
oof_auc = None
if sorted(FOLDS) == list(range(split.attrs["n_folds"])):
    oof = pd.concat([pd.read_csv(RESULTS / f"oof_f{f}.csv") for f in FOLDS]).merge(pool[["StudyInstanceUID", *LABELS]], on="StudyInstanceUID")
    oof_auc = mean_auc(oof[LABELS].to_numpy(), oof[[f"pred_{l}" for l in LABELS]].to_numpy())["mean"]

parent = (leer_metricas(PADRE) if PADRE else None) or {}
parent_f0 = parent.get("per_fold", {}).get("0", {}).get("pseudo_auc")
ours_f0 = per_fold.get("0", {}).get("pseudo_auc")
delta_pseudo = ours_f0 - parent_f0 if ours_f0 is not None and parent_f0 is not None else None
# screen (solo fold 0): gold contra el gold del fold 0 del padre, no contra su promedio de 5 folds
parent_gold = parent.get("per_fold", {}).get("0", {}).get("gold_auc") if FOLDS == [0] else parent.get("gold_auc")
delta_gold = gold_auc["mean"] - parent_gold if parent_gold is not None else None
DECISION = decision(delta_pseudo, delta_gold)

summary = pd.DataFrame({
    "padre": [parent_f0, parent_gold, parent.get("oof_pseudo_auc")],
    "este": [ours_f0, gold_auc["mean"], oof_auc],
}, index=["pseudo fold 0", "gold (58)", "OOF pseudo 5 folds"])
summary["delta"] = summary.este - summary.padre
display(summary.round(4))
display(pd.DataFrame({f: v["gold_per_label"] for f, v in per_fold.items()}).round(3))
print("Decisión automática:", DECISION)
for f in FOLDS:
    pd.read_csv(RESULTS / f"history_f{f}.csv").plot(x="epoch", y=["pseudo_auc", "gold_auc"], marker="o", title=f"{EXP} fold {f}")
    plt.show()

## 6. Guardar `results/<EXP>/metrics.json`

In [ ]:
metrics = {
    "schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE,
    "split": SPLIT, "split_sha256": split.attrs["sha256"], "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else ("confirm" if oof_auc is not None else "screen"),
    "select_on": SELECT_ON, "params": PARAMS, "per_fold": per_fold,
    "gold_auc": gold_auc["mean"], "gold_per_label": gold_auc["per_label"], "oof_pseudo_auc": oof_auc,
    "delta": {"pseudo_f0": delta_pseudo, "gold": delta_gold},
    "decision_auto": DECISION,
    "gpu_hours_total": sum(f["gpu_hours"] for f in per_fold.values()),
}
print("Guardado:", guardar_metricas(RESULTS, metrics))

## 7. Verdict

Copia los números de la celda 5 a la sección 3 de `SPEC.md` y actualiza tu fila en `docs/EXPERIMENTOS.md`
(estado `worked`, `didnt` o `killed` y una línea de qué aprendimos). Después: commit y PR a `main`.